<a href="https://colab.research.google.com/github/walkea14/duckietown-lx/blob/DTSW-2346-LX-DD21-7-Localization/Module04_ProgrammingCode_AlexWalker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install tensorflow==2.15.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.2
    Uninstalling wrapt-1.17.2:
      Successfully uninstalled wrapt-1.17.2
  Attempting uninstall: ml-dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0
  Attempting uninstall: tensorboard
    Found existing installation

In [2]:
#@title Data Retrieval
#Download CSV file named electrical_grid_stability_simulated_data.csv from GitHub repository using wget
!wget https://raw.githubusercontent.com/mhrafiei/data/main/electrical_grid_stability_simulated_data.csv

--2025-02-15 09:38:03--  https://raw.githubusercontent.com/mhrafiei/data/main/electrical_grid_stability_simulated_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2417871 (2.3M) [text/plain]
Saving to: ‘electrical_grid_stability_simulated_data.csv’

electrical_grid_sta 100%[===================>]   2.31M  --.-KB/s    in 0.06s   

2025-02-15 09:38:03 (39.8 MB/s) - ‘electrical_grid_stability_simulated_data.csv’ saved [2417871/2417871]



In [3]:
#@title Mount Google Drive and Create Result Folder
from google.colab import drive
drive.mount('/content/drive')

!mkdir /content/drive/MyDrive/RESULTS_535742
!mkdir /content/drive/MyDrive/RESULTS_535742/Module_04/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mkdir: cannot create directory ‘/content/drive/MyDrive/RESULTS_535742’: File exists
mkdir: cannot create directory ‘/content/drive/MyDrive/RESULTS_535742/Module_04/’: File exists


In [4]:
#@title Import Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import random
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error as fun_mse


In [5]:
#@title Data Processing, Plot, and Machine Learning Functions
'''datain = data.iloc[:,0:12]
dataou = data.iloc[:, 12:13]
dataou.head()

rtt = [0.1, 0.3]
rrs = 3'''

def fun_data_processing(datain, dataou, rtt, random_integer):
    # Split the input data and output labels into training and testing sets using scikit-learn's train_test_split function.
    datain_train, datain_test, dataou_train, dataou_test = train_test_split(datain, dataou, test_size=rtt)
    datain_train.head()
    dataou_train.head()
      # 'rtt' is used to specify the proportion of the dataset to include in the test split.
    rtt = rtt
      # 'random_state' ensures reproducibility; each time this function is called, we need to have a different integer for 'random_state' (why?)

      # Create an instance of the MinMaxScaler, specifying the feature range to normalize the input data between 0 and 1.
    scaler_in = MinMaxScaler(feature_range=(0,1))

      # Create an instance of the MinMaxScaler, specifying the feature range to normalize the output data between 0 and 1.
    scaler_ou = MinMaxScaler(feature_range=(0,1))

      # Fit the scalers on the training data only to prevent data leakage from the test set.
    datain_train_scaled = scaler_in.fit_transform(datain_train)
    dataou_train_scaled = scaler_ou.fit_transform(dataou_train)
      # Transform the training and testing input and output data using the fitted scalers.

      # Store the scaled input data and original output labels in a dictionary to organize and simplify data handling.
    data_scaled = {} # Create an empty dictionary and then keep creating key values.
    data_scaled['in_tr'] = datain_train_scaled
    data_scaled['in_te'] = datain_test
    data_scaled['ou_tr'] = dataou_train_scaled
    data_scaled['ou_te'] = dataou_test
    #data_scaled = {'in_tr': datain_train_scaled, 'in_te': datain_test, 'ou_tr': dataou_train_scaled, 'ou_te': dataou_test}

      # Return the dictionary containing all scaled data and the scalers (optional).



    return data_scaled




In [6]:
#@title Plot function
'''
This function is to generate required plots once gettin the training/testing.
***This cell was copied from Module04_Assignment_Hints***
'''
def fun_plot(result, rtt, rrs):
    # Plotting function to visualize the performance and results of the machine learning model.

    # 1. Plot for Training Data: Real vs Estimated values
    plt.figure(figsize=[8, 8])  # Set the figure size to 8x8 inches.

    # Scatter plot of actual vs predicted values with small markers.
    plt.plot(result['ou_tr'], result['es_tr'], '.', markersize=1)

    # Plot a red line (bi-sector) representing the ideal case where real values equal estimated values.
    plt.plot([0, 1], [0, 1], '-r', linewidth=2)

    # Setting labels and title with customized font sizes.
    plt.xlabel('Real', fontsize=20)
    plt.ylabel('Estimation', fontsize=20)

    # Title includes RTT (Ratio of Training-Testing) and RRS (Repeated Random Sampling) with formatted values.
    plt.title('TR | RTT {:4.2f} | RRS {:02d}'.format(rtt, rrs), fontsize=20)

    # Save the figure to a PNG file named after the RTT and RRS values.
    plt.savefig('real_vs_es_TR_{:02d}_{:02d}.png'.format(int(rtt*100), rrs))

    # Close the plot to free up memory.
    plt.close()

    # 2. Plot for Testing Data: Real vs Estimated values
    plt.figure(figsize=[8, 8])  # Similar setup for the testing data plot.

    # Scatter plot of actual vs predicted values on testing data.
    plt.plot(result['ou_te'], result['es_te'], '.', markersize=1)

    # Ideal line (bi-sector) for reference.
    plt.plot([0, 1], [0, 1], '-r', linewidth=2)

    # Setting labels and title with customized font sizes for the testing data plot.
    plt.xlabel('Real', fontsize=20)
    plt.ylabel('Estimation', fontsize=20)
    plt.title('TE | RTT {:4.2f} | RRS {:02d}'.format(rtt, rrs), fontsize=20)

    # Save the figure to a PNG file for testing data, named similarly with RTT and RRS values.
    plt.savefig('real_vs_es_TE_{:02d}_{:02d}.png'.format(int(rtt*100), rrs))

    plt.close()

    # 3. Loss Plots for Training and Validation
    plt.figure(figsize=[8, 8])  # Setting figure size for the loss plot.

    # Plotting the natural log of the training and validation loss to better visualize changes over epochs.
    plt.plot(np.log(result['ls_tr']), '-', linewidth=2)
    plt.plot(np.log(result['ls_vl']), '--', linewidth=2)

    # Setting labels and title with font size customization for the loss plot.
    plt.xlabel("Epochs", fontsize=20)
    plt.ylabel("Natural Log of Loss", fontsize=20)
    plt.title("Epoch VS Natural Log of Losses | RTT {:4.2f} | RRS {:02d}".format(rtt, rrs), fontsize=20)

    # Adding a legend to distinguish between training and validation losses.
    plt.legend(['TR Loss', 'VL Loss'], fontsize=20)

    # Save the loss plot to a PNG file, named after RTT and RRS values.
    plt.savefig('epoch_naturallog_of_losses_{:02d}_{:02d}.png'.format(int(rtt*100), rrs))

    plt.close()
    # This function creates and saves three types of plots:
    # - The first two plots compare the real output values to the estimated values from the model for both training and testing datasets.
    # - The third plot shows the natural logarithm of the loss over epochs (also called learning plots)
    # for both training and validation sets, providing insights into the learning process.


In [29]:
#@title Machine Learning Model

def fun_ml(data):
  #Define input layer with
  input = tf.keras.Input(shape=(data['in_tr'].shape[1],))

  #First hidden layer with 75 neurons, applying ReLU activation function after the dense layer and 10% dropout layer
  x = tf.keras.layers.Dense(75)
  x = tf.keras.layers.ReLU()(x)
  x = tf.keras.layers.Dropout(0.1)(x)

  #Second hidden layer with 50 neurons, ReLu activation function and 10% dropout layer
  x = tf.keras.layers.Dense(50)
  x = tf.keras.layers.ReLU()(x)
  x = tf.keras.layers.Dropout(0.1)(x)

  #Third hidden layer with 25 neurons, ReLU activation and 10% dropout layer
  x = tf.keras.layers.Dense(25)
  x = tf.keras.layers.ReLU()(x)
  x = tf.keras.layers.Dropout(0.1)(x)

  #Final hidden layer with 10 neurons, ReLU activation and 10% dropout layer
  x = tf.keras.layers.Dense(10)
  x = tf.keras.layers.ReLU()(x)
  x = tf.keras.layers.Dropout(0.1)(x)

  #Output layer with single neuron for regression output
  output = tf.keras.layers.Dense(1)(x)

  #Create the model with input and output layers
  model = tf.keras.Model(input, output)

  #Compiling model with the Adam optimizer and mean squared error as loss function
  model.compile(optimizer='adam', loss='mse')

  #Train model with training data, specifying epochs, batch size, verbosity level, validation split, and shuffle
  history = model.fit(data['in_tr'], data['ou_tr'], epochs=100, batch_size=32, verbose=0, validation_split=0.2, shuffle=True)

  #Gather predictions for both training and testing datasets
  results = {}
  results['es_tr'] = model.predict(data['in_tr'], verbose=3) #estimate of training input
  results['es_te'] = model.predict(data['in_te'], verbose=3) #estimate of testing input
  results['ou_tr'] = data['ou_tr'] #Actual output training data
  results['ou_te'] = data['ou_te'] #Actual output testing data

  #Calculate mean squared error for both training and testing predictions compared to actual output
  results['mse_tr'] = fun_mse(results['ou_tr'], results['es_tr'])
  results['mse_te'] = fun_mse(results['ou_te'], results['es_te'])

  #Storing loss history for both training and validation for further analysis
  results['ls_tr'] = history.history['loss']
  results['ls_vl'] = history.history['val_loss']

  return results

In [30]:
from inspect import EndOfBlock
#@title Main Run
RTT = [0.1, 0.3] #Ratio of Training to Testing
RRS = 3 #Repetitions per RTT

#load dataset from CSV into Pandas DataFrame
df = pd.read_csv('electrical_grid_stability_simulated_data.csv')

#Select input features and output target from DataFrame
datain = df.iloc[:,:12] #Input
dataou = df.iloc[:,-2:-1] #Output

#Generate random integers to use as 'random_state' argument
integers = np.random.randint(25, 1000, int(len(RTT)*RRS))

#Initialize empty array to store results of model fro each RTT and repetition
results = np.empty((len(RTT), RRS), dtype=dict)

#Counter to iterate through the generated random integers
counter = 0

#Loop over each RTT for each repetition
for i, rtt in enumerate(RTT):
  print(i)
  for rrs in range(RRS):
    #Process data by splitting into training and testing sets, then scaling it
    data = fun_data_processing(datain, dataou, rtt, integers[counter])
    #Train machine learning model using processed data and store results
    results[i, rrs] = fun_ml(data)

    #Plot results for current RTT and repetition
    fun_plot(results[i, rrs], rtt, rrs+1)

    #Print MSE for training and testing datasets along with current RTT and reptition
    print("RTT {} | RRS {:02d} | MSE TR {:6.4f}".format(rtt, rrs+1, results[i, rrs]['mse_tr'], results[i,rrs]['mse_te']))

    #Increment counter
    counter = counter + 1


0


ValueError: Only input tensors may be passed as positional arguments. The following argument value should be passed as a keyword argument: <Dense name=dense_2, built=False> (of type <class 'keras.src.layers.core.dense.Dense'>)

In [ ]:
#@title Table of MSEs

#Initialize empty dictionary to store MSE results organized by experiment number
df = {}

#Iterate through each repeated random sample (RRS)
for i in range (RRS):
  #Initialize empty list for each repetition to store MSE info
  df[f'{i + 1:02d}'] = []
  for j in range(len(RTT))
    #Format training and testing MSEs for model for current config
    #Format MSEs to four decimal places
    cell_info = f"{results_X[j,i]['acc_tr'] :7.4f} ({results_X[j,i]['acc_te'] : 7.4f})"

    #Append MSE info to lisst corresponding to current repetition
    df[f'{i + 1:02d}'].append(cell_info)
#Convert dictionary to DataFrame where columns represent different repetitions of random samplings and rows represent different test set ratios
df = pd.DataFrame(df)

#Export DataFrame to CSV file, storing table of MSEs for further analysis or reporting
df.to_csv('Table_mses.csv')

In [ ]:
#@title Zip Results adn Transfer to Google Drive Designated Folder
!zip results.zip *.png *.csv

!cp results.zip /content/drive/MyDrive/RESULTS_535742/Module_04/results.zip

In [ ]:
#@title Auto runtime disconnection to save CPU/GPU/TPU allocations

from google.colab import runtime
import time

time.sleep(60)

runtime.unassign()